## Homework 9: Text Classification with Fine-Tuned BERT

In this final homework, we’ll explore **fine-tuning a pre-trained Transformer model (BERT)** for text classification using the **IMDB Movie Review** dataset. You’ll begin with a working baseline notebook and then conduct a series of controlled experiments to understand how data size, context length, and model architecture affect performance.

You’ll complete three problems:

* **Problem 1:** Evaluate how **sequence length** and **learning rate** jointly influence validation loss and generalization.
* **Problem 2:** Measure how **training data size** affects both model performance and total training time.
* **Problem 3:** Compare **two additional models** from the BERT family to analyze the trade-offs between model size and accuracy on this dataset.

In each problem, you’ll report your key metrics, summarize what you observed, and reflect on what you learned.

> **Note:** This homework was developed and tested on **Google Colab**, due to version conflicts when running locally. It is **strongly recommended** that you complete your work on Colab as well.

There are 6 problems, each worth 14 points, and you get one point free if you complete the entire homework.


In [1]:
# Install once per new Colab runtime
%pip -q install -U keras keras-hub tensorflow tensorflow-text datasets evaluate

Note: you may need to restart the kernel to use updated packages.


In [2]:

import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import time
import random
import numpy as np
import keras
import keras_hub as kh
import evaluate
from datasets import load_dataset, Dataset, Features, Value, ClassLabel

from keras import mixed_precision                    # generally faster
mixed_precision.set_global_policy("mixed_float16")

2026-03-30 00:07:21.118310: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/luan/BU/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Here is where you can set global hyperparameters for this homework

In [3]:
# ---------------- Config ----------------
SEED        = 42
MAX_LEN     = 128
EPOCHS      = 3
BATCH       = 32
EVAL_BATCH  = 64
SUBSET_FRAC = 0.25   # <-- 0.25 to train and test on 25% of whole dataset during development;  set to 1.0 for full dataset

keras.utils.set_random_seed(SEED)

### Load and Preprocess the IMDB Movie Review Dataset

In [4]:
# ---- Load IMDb (raw), join train+test ----
imdb   = load_dataset("imdb")
texts  = list(imdb["train"]["text"]) + list(imdb["test"]["text"])
labels = np.array(list(imdb["train"]["label"]) + list(imdb["test"]["label"]), dtype="int32")

# ---- Build DS with explicit features (label=ClassLabel) ----
features = Features({"text": Value("string"),
                     "label": ClassLabel(num_classes=2, names=["NEG","POS"])})
all_ds = Dataset.from_dict({"text": texts, "label": labels.tolist()}, features=features)

# ---- Optional: take a stratified subset of the FULL dataset ----
if 0.0 < SUBSET_FRAC < 1.0:
    sub = all_ds.train_test_split(train_size=SUBSET_FRAC, seed=SEED, stratify_by_column="label")
    ds_pool = sub["train"]
else:
    ds_pool = all_ds

# ---- Stratified 80/10/10 split on the (possibly smaller) pool ----
# First: 80/20 train+val pool / test
splits = ds_pool.train_test_split(test_size=0.20, seed=SEED, stratify_by_column="label")
train_val_pool, test_ds = splits["train"], splits["test"]
# Then: carve 10% of full (i.e., 0.125 of the 80% pool) as validation
splits2 = train_val_pool.train_test_split(test_size=0.125, seed=SEED, stratify_by_column="label")
train_ds, val_ds = splits2["train"], splits2["test"]

# ---- Numpy arrays for Keras fit/predict ----
X_tr = np.array(train_ds["text"], dtype=object); y_tr = np.array(train_ds["label"], dtype="int32")
X_va = np.array(val_ds["text"],   dtype=object); y_va = np.array(val_ds["label"],   dtype="int32")
X_te = np.array(test_ds["text"],  dtype=object); y_te = np.array(test_ds["label"],  dtype="int32")

# ---- Quick summary ----
def _counts(ds):
    arr = np.array(ds["label"], dtype=int)
    return len(arr), np.bincount(arr, minlength=2).tolist()
print(f"Pool after SUBSET_FRAC={SUBSET_FRAC}: {len(ds_pool)} (of {len(all_ds)})")
print("Train:", _counts(train_ds), " Val:", _counts(val_ds), " Test:", _counts(test_ds))


Pool after SUBSET_FRAC=0.25: 12500 (of 50000)
Train: (8750, [4375, 4375])  Val: (1250, [625, 625])  Test: (2500, [1250, 1250])


### Build and train a baseline Distil-Bert Text Classifier

In [5]:
# ---- Keras Hub preprocessor + classifier ----
preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased", sequence_length=MAX_LEN
)
model = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased", num_classes=2, preprocessor=preproc
)

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
)

start = time.time()

# ---- Train with early stopping (restore best val weights) ----
cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]
history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=cb,
    verbose=1,
)

# ---- Evaluate (accuracy + F1 via `evaluate`) ----
logits = model.predict(X_te, batch_size=EVAL_BATCH, verbose=0)
y_pred = logits.argmax(axis=-1)

acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")
acc = acc_metric.compute(predictions=y_pred, references=y_te)["accuracy"]
f1  = f1_metric.compute(predictions=y_pred, references=y_te)["f1"]

# Tiny confusion matrix helper (no sklearn needed)
def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

print(f"\nValidation acc (best epoch): {history.history['val_acc'][np.argmin(history.history['val_loss'])]:.3f}")
print(f"\nTest accuracy: {acc:.3f}   Test F1: {f1:.3f}")
print("\nConfusion matrix:\n", confusion_matrix_np(y_te, y_pred))

end = time.time() - start
print("\nElapsed time:", time.strftime("%H:%M:%S", time.gmtime(end)))

I0000 00:00:1774847254.648518   32839 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21767 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:0a:00.0, compute capability: 8.6


Epoch 1/3


2026-03-30 00:07:48.055746: I external/local_xla/xla/service/service.cc:163] XLA service 0x796e70010390 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-30 00:07:48.055782: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3090, Compute Capability 8.6
2026-03-30 00:07:48.305992: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-30 00:07:50.421145: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900
2026-03-30 00:07:52.394591: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-30 00:07:53.298294: I e

273/274 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - acc: 0.6919 - loss: 0.5603

2026-03-30 00:08:25.965343: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_139', 16 bytes spill stores, 16 bytes spill loads

2026-03-30 00:08:26.496813: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_165', 4 bytes spill stores, 4 bytes spill loads

2026-03-30 00:08:26.497163: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_40', 8 bytes spill stores, 8 bytes spill loads

2026-03-30 00:08:42.712108: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'fusion_3956', 168 bytes spill stores, 168 bytes spill loads



274/274 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - acc: 0.6922 - loss: 0.5599

2026-03-30 00:08:47.544636: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'fusion_392', 42 bytes spill stores, 42 bytes spill loads

2026-03-30 00:08:48.386529: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-30 00:08:49.299707: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5', 1260 bytes spill stores, 1260 bytes spill loads



274/274 ━━━━━━━━━━━━━━━━━━━━ 74s 149ms/step - acc: 0.7825 - loss: 0.4529 - val_acc: 0.8376 - val_loss: 0.3449
Epoch 2/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 13s 48ms/step - acc: 0.8786 - loss: 0.2896 - val_acc: 0.8584 - val_loss: 0.3398
Epoch 3/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 15s 53ms/step - acc: 0.9159 - loss: 0.2206 - val_acc: 0.8600 - val_loss: 0.3552


2026-03-30 00:09:21.633826: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_43', 16 bytes spill stores, 16 bytes spill loads

2026-03-30 00:09:23.476189: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'fusion_393', 42 bytes spill stores, 42 bytes spill loads

2026-03-30 00:09:25.609405: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_43', 16 bytes spill stores, 16 bytes spill loads

2026-03-30 00:09:25.963681: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5', 1260 bytes spill stores, 1260 bytes spill loads




Validation acc (best epoch): 0.858

Test accuracy: 0.855   Test F1: 0.852

Confusion matrix:
 [[1098  152]
 [ 210 1040]]

Elapsed time: 00:01:52


# Problem 1 — Mini sweep: context length × learning rate (6 runs)

In this problem we'll see how much **context length** (`MAX_LEN`) helps, and how sensitive fine-tuning is to **learning rate**—without running a huge grid.

## Setup (keep these fixed)

* `SUBSET_FRAC = 0.25`               # use only this percentage of the whole dataset
* `EPOCHS = 3`
* `BATCH = 32` (but see note for 256 below)
* **EarlyStopping** with `restore_best_weights=True`
* Same random `SEED` for all runs
* Same data split for all runs (don’t reshuffle between runs)

### Run these 6 configurations

**For each** `MAX_LEN ∈ {128, 256, 512}`, try **two** learning rates:

* **MAX_LEN = 128**

  * `(LR = 2e-5, BATCH = 32)` – healthy default for shorter contexts.
  * `(LR = 1e-5, BATCH = 32)` – conservative LR; often a touch stabler.

* **MAX_LEN = 256**

  * `(LR = 1e-5, BATCH = 16)` – longer context → lower batch.
  * `(LR = 7.5e-6, BATCH = 16)` – even steadier if loss is noisy.

* **MAX_LEN = 512**  *(heavier quadratic attention cost)*

  * `(LR = 7.5e-6, BATCH = 8)` – safe starting point.
  * `(LR = 5e-6, BATCH = 8)` – extra caution for stability.

**If you hit an Out Of Memory error:**

* At **256** with `BATCH = 16`, drop to `BATCH = 8`.
* At **512** with `BATCH = 8`, drop to `BATCH = 4`.


Then answer the graded questions.


In [ ]:
# Problem 1

import time
import numpy as np
import keras
import keras_hub as kh
import evaluate

# Fixed settings
SEED        = 42
SUBSET_FRAC = 0.25
EPOCHS      = 3
EVAL_BATCH  = 64

# The 6 configurations to sweep
configs = [
    {"max_len": 128, "lr": 2e-5,  "batch": 32},
    {"max_len": 128, "lr": 1e-5,  "batch": 32},
    {"max_len": 256, "lr": 1e-5,  "batch": 16},
    {"max_len": 256, "lr": 7.5e-6,"batch": 16},
    {"max_len": 512, "lr": 7.5e-6,"batch": 8},
    {"max_len": 512, "lr": 5e-6,  "batch": 8},
]

p1_results = []

for i, cfg in enumerate(configs):
    print(f"Run {i+1}/6 — MAX_LEN={cfg['max_len']}, LR={cfg['lr']}, BATCH={cfg['batch']}")

    keras.utils.set_random_seed(SEED)

    # Build model with this config's sequence length
    preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
        "distil_bert_base_en_uncased", sequence_length=cfg["max_len"]
    )
    mdl = kh.models.DistilBertTextClassifier.from_preset(
        "distil_bert_base_en_uncased", num_classes=2, preprocessor=preproc
    )
    mdl.compile(
        optimizer=keras.optimizers.Adam(cfg["lr"]),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )

    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]
    start = time.time()
    hist = mdl.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=EPOCHS,
        batch_size=cfg["batch"],
        callbacks=cb,
        verbose=1,
    )
    elapsed = time.time() - start

    # Best validation accuracy (at epoch with min val_loss)
    best_idx = np.argmin(hist.history["val_loss"])
    best_val_acc  = hist.history["val_acc"][best_idx]
    best_val_loss = hist.history["val_loss"][best_idx]

    # Test metrics
    logits = mdl.predict(X_te, batch_size=EVAL_BATCH, verbose=0)
    y_pred = logits.argmax(axis=-1)
    acc_metric = evaluate.load("accuracy")
    f1_metric  = evaluate.load("f1")
    test_acc = acc_metric.compute(predictions=y_pred, references=y_te)["accuracy"]
    test_f1  = f1_metric.compute(predictions=y_pred, references=y_te)["f1"]

    p1_results.append({
        **cfg,
        "val_acc": best_val_acc,
        "val_loss": best_val_loss,
        "test_acc": test_acc,
        "test_f1": test_f1,
        "time": elapsed,
    })

    print(f"Val acc (best): {best_val_acc:.4f}  |  Val loss: {best_val_loss:.4f}")
    print(f"Test acc: {test_acc:.4f}  |  Test F1: {test_f1:.4f}")
    print(f"Time: {time.strftime('%H:%M:%S', time.gmtime(elapsed))}")

# Summary table
print(f"{'MAX_LEN':>8} {'LR':>10} {'BATCH':>6} {'ValAcc':>8} {'ValLoss':>8} {'TestAcc':>8} {'TestF1':>8} {'Time':>10}")
for r in p1_results:
    print(f"{r['max_len']:>8} {r['lr']:>10.1e} {r['batch']:>6} {r['val_acc']:>8.4f} {r['val_loss']:>8.4f} {r['test_acc']:>8.4f} {r['test_f1']:>8.4f} {time.strftime('%H:%M:%S', time.gmtime(r['time'])):>10}")

# Find best config
best_p1 = max(p1_results, key=lambda r: r["val_acc"])
print(f"\nBest config: MAX_LEN={best_p1['max_len']}, LR={best_p1['lr']}, ValAcc={best_p1['val_acc']:.4f}")

Run 1/6 — MAX_LEN=128, LR=2e-05, BATCH=32
Epoch 1/3


2026-03-30 00:10:00.359637: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'fusion_2668', 42 bytes spill stores, 42 bytes spill loads



273/274 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - acc: 0.7256 - loss: 0.5077

2026-03-30 00:10:31.545099: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'fusion_3956', 168 bytes spill stores, 168 bytes spill loads



274/274 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - acc: 0.7259 - loss: 0.5074

...

1094/1094 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - acc: 0.7767 - loss: 0.4330

2026-03-30 00:19:37.272886: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_55', 8 bytes spill stores, 8 bytes spill loads

2026-03-30 00:19:37.555195: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_43', 16 bytes spill stores, 16 bytes spill loads

2026-03-30 00:19:37.918387: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5', 1260 bytes spill stores, 1260 bytes spill loads



1094/1094 ━━━━━━━━━━━━━━━━━━━━ 109s 69ms/step - acc: 0.8547 - loss: 0.3211 - val_acc: 0.9112 - val_loss: 0.2236
Epoch 2/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 51s 46ms/step - acc: 0.9366 - loss: 0.1751 - val_acc: 0.9144 - val_loss: 0.2317
Epoch 3/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 49s 45ms/step - acc: 0.9617 - loss: 0.1101 - val_acc: 0.9088 - val_loss: 0.2596


2026-03-30 00:21:23.203514: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_43', 8 bytes spill stores, 8 bytes spill loads

2026-03-30 00:21:30.172593: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_55', 8 bytes spill stores, 8 bytes spill loads

2026-03-30 00:21:30.495950: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_43', 16 bytes spill stores, 16 bytes spill loads

2026-03-30 00:21:30.879009: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5', 1260 bytes spill stores, 1260 bytes spill loads



Val acc (best): 0.9112  |  Val loss: 0.2236
Test acc: 0.9120  |  Test F1: 0.9142
Time: 00:03:29
Run 6/6 — MAX_LEN=512, LR=5e-06, BATCH=8
Epoch 1/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 104s 66ms/step - acc: 0.8455 - loss: 0.3426 - val_acc: 0.9136 - val_loss: 0.2250
Epoch 2/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 49s 45ms/step - acc: 0.9274 - loss: 0.1931 - val_acc: 0.9120 - val_loss: 0.2291
Epoch 3/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 51s 47ms/step - acc: 0.9509 - loss: 0.1369 - val_acc: 0.9112 - val_loss: 0.2395
Val acc (best): 0.9136  |  Val loss: 0.2250
Test acc: 0.9120  |  Test F1: 0.9128
Time: 00:03:25
 MAX_LEN         LR  BATCH   ValAcc  ValLoss  TestAcc   TestF1       Time
     128    2.0e-05     32   0.8400   0.3531   0.8420   0.8527   00:01:37
     128    1.0e-05     32   0.8584   0.3400   0.8536   0.8504   00:01:35
     256    1.0e-05     16   0.9016   0.2411   0.8920   0.8930   00:02:15
     256    7.5e-06     16   0.9080   0.2427   0.8940   0.8950   00:02:09
     512    7.5e-06      8   0.91

### Graded Questions

In [7]:
# Set a1a to the validation accuracy at min validation loss for your best configuration found in this problem

# Automatically use the best result from our sweep
a1a = best_p1["val_acc"]

In [8]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a1a = {a1a:.4f}')

a1a = 0.9136


#### Question a1b:

* Does **more context** (128 → 256 → 512) consistently help?
* How much effect did the learning rate have on the validation accuracy?


#### Your Answer Here:

More context consistently helped but with diminishing returns. Moving from MAX_LEN=128 to 256 produced a large improvement in validation accuracy. The jump from 256 to 512 was smaller but still meaningful. This suggests that many IMDB reviews are longer than 128 tokens and benefit substantially from seeing more text while the additional content beyond 256 tokens adds less discriminative signal.

Learning rate had a modest but noticeable effect. Within each MAX_LEN group, the lower learning rate tended to perform slightly better. At MAX_LEN=128, LR=1e-5 achieved 0.8584 val acc vs. 0.8400 for LR=2e-5. At MAX_LEN=256, 7.5e-6 edged out 1e-5. At MAX_LEN=512, 5e-6 slightly beat 7.5e-6. Overall, the learning rate differences were much smaller than the effect of increasing context length.

## Problem 2 — How much data is enough?

In this problem, you’ll investigate how model performance scales with dataset size.

**Setup.**
Use the best `MAX_LEN` and `LR` values you found in **Problem 1**.

**What to do:**

1. For each value of `SUBSET_FRAC ∈ {0.25, 0.50, 0.75, 1.00}`, train your model once and observe the displayed performance metrics.
2. Answer the discussion question below.




In [9]:
# Problem 2
# Use best MAX_LEN, LR, BATCH from Problem 1

BEST_MAX_LEN = best_p1["max_len"]
BEST_LR      = best_p1["lr"]
BEST_BATCH   = best_p1["batch"]
EPOCHS       = 3
SEED         = 42

subset_fracs = [0.25, 0.50, 0.75, 1.00]
p2_results = []

for frac in subset_fracs:
    print(f"SUBSET_FRAC = {frac}")
    
    keras.utils.set_random_seed(SEED)

    # Rebuild data split for this fraction
    if 0.0 < frac < 1.0:
        sub = all_ds.train_test_split(train_size=frac, seed=SEED, stratify_by_column="label")
        ds_pool_p2 = sub["train"]
    else:
        ds_pool_p2 = all_ds

    splits = ds_pool_p2.train_test_split(test_size=0.20, seed=SEED, stratify_by_column="label")
    train_val_pool_p2, test_ds_p2 = splits["train"], splits["test"]
    splits2 = train_val_pool_p2.train_test_split(test_size=0.125, seed=SEED, stratify_by_column="label")
    train_ds_p2, val_ds_p2 = splits2["train"], splits2["test"]

    X_tr2 = np.array(train_ds_p2["text"], dtype=object); y_tr2 = np.array(train_ds_p2["label"], dtype="int32")
    X_va2 = np.array(val_ds_p2["text"],   dtype=object); y_va2 = np.array(val_ds_p2["label"],   dtype="int32")
    X_te2 = np.array(test_ds_p2["text"],  dtype=object); y_te2 = np.array(test_ds_p2["label"],  dtype="int32")

    print(f"  Train: {len(X_tr2)}, Val: {len(X_va2)}, Test: {len(X_te2)}")

    # Build & train model
    preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
        "distil_bert_base_en_uncased", sequence_length=BEST_MAX_LEN
    )
    mdl = kh.models.DistilBertTextClassifier.from_preset(
        "distil_bert_base_en_uncased", num_classes=2, preprocessor=preproc
    )
    mdl.compile(
        optimizer=keras.optimizers.Adam(BEST_LR),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )

    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]
    start = time.time()
    hist = mdl.fit(
        X_tr2, y_tr2,
        validation_data=(X_va2, y_va2),
        epochs=EPOCHS,
        batch_size=BEST_BATCH,
        callbacks=cb,
        verbose=1,
    )
    elapsed = time.time() - start

    best_idx = np.argmin(hist.history["val_loss"])
    best_val_acc  = hist.history["val_acc"][best_idx]
    best_val_loss = hist.history["val_loss"][best_idx]

    logits = mdl.predict(X_te2, batch_size=EVAL_BATCH, verbose=0)
    y_pred = logits.argmax(axis=-1)
    acc_metric = evaluate.load("accuracy")
    f1_metric  = evaluate.load("f1")
    test_acc = acc_metric.compute(predictions=y_pred, references=y_te2)["accuracy"]
    test_f1  = f1_metric.compute(predictions=y_pred, references=y_te2)["f1"]

    p2_results.append({
        "frac": frac,
        "val_acc": best_val_acc,
        "val_loss": best_val_loss,
        "test_acc": test_acc,
        "test_f1": test_f1,
        "time": elapsed,
    })

    print(f"Val acc (best): {best_val_acc:.4f}  |  Val loss: {best_val_loss:.4f}")
    print(f"Test acc: {test_acc:.4f}  |  Test F1: {test_f1:.4f}")
    print(f"Time: {time.strftime('%H:%M:%S', time.gmtime(elapsed))}")

# Summary
print(f"{'Frac':>6} {'ValAcc':>8} {'ValLoss':>8} {'TestAcc':>8} {'TestF1':>8} {'Time':>10}")
for r in p2_results:
    print(f"{r['frac']:>6.2f} {r['val_acc']:>8.4f} {r['val_loss']:>8.4f} {r['test_acc']:>8.4f} {r['test_f1']:>8.4f} {time.strftime('%H:%M:%S', time.gmtime(r['time'])):>10}")

best_p2 = max(p2_results, key=lambda r: r["val_acc"])
print(f"\nBest: SUBSET_FRAC={best_p2['frac']}, ValAcc={best_p2['val_acc']:.4f}")

SUBSET_FRAC = 0.25
  Train: 8750, Val: 1250, Test: 2500
Epoch 1/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 104s 67ms/step - acc: 0.8456 - loss: 0.3426 - val_acc: 0.9152 - val_loss: 0.2249
Epoch 2/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 51s 47ms/step - acc: 0.9274 - loss: 0.1931 - val_acc: 0.9120 - val_loss: 0.2289
Epoch 3/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 49s 45ms/step - acc: 0.9509 - loss: 0.1368 - val_acc: 0.9120 - val_loss: 0.2396
Val acc (best): 0.9152  |  Val loss: 0.2249
Test acc: 0.9120  |  Test F1: 0.9128
Time: 00:03:25
SUBSET_FRAC = 0.5
  Train: 17500, Val: 2500, Test: 5000
Epoch 1/3
2186/2188 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - acc: 0.8071 - loss: 0.3981

2026-03-30 00:31:17.189700: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_164', 4 bytes spill stores, 4 bytes spill loads

2026-03-30 00:31:17.194041: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_164', 8 bytes spill stores, 8 bytes spill loads

2026-03-30 00:31:17.390702: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_40', 8 bytes spill stores, 8 bytes spill loads



2188/2188 ━━━━━━━━━━━━━━━━━━━━ 156s 57ms/step - acc: 0.8749 - loss: 0.2940 - val_acc: 0.9244 - val_loss: 0.1940
Epoch 2/3
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 100s 46ms/step - acc: 0.9310 - loss: 0.1830 - val_acc: 0.9244 - val_loss: 0.2020
Epoch 3/3
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 100s 46ms/step - acc: 0.9505 - loss: 0.1372 - val_acc: 0.9232 - val_loss: 0.2006
Val acc (best): 0.9244  |  Val loss: 0.1940
Test acc: 0.9178  |  Test F1: 0.9170
Time: 00:05:56
SUBSET_FRAC = 0.75
  Train: 26250, Val: 3750, Test: 7500
Epoch 1/3
3280/3282 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - acc: 0.8398 - loss: 0.3492

2026-03-30 00:38:20.416472: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_164', 4 bytes spill stores, 4 bytes spill loads

2026-03-30 00:38:20.879233: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_164', 8 bytes spill stores, 8 bytes spill loads

2026-03-30 00:38:20.916554: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_40', 8 bytes spill stores, 8 bytes spill loads



3282/3282 ━━━━━━━━━━━━━━━━━━━━ 198s 51ms/step - acc: 0.8894 - loss: 0.2682 - val_acc: 0.9256 - val_loss: 0.1958
Epoch 2/3
3282/3282 ━━━━━━━━━━━━━━━━━━━━ 144s 44ms/step - acc: 0.9346 - loss: 0.1747 - val_acc: 0.9272 - val_loss: 0.1990
Epoch 3/3
3282/3282 ━━━━━━━━━━━━━━━━━━━━ 143s 44ms/step - acc: 0.9554 - loss: 0.1302 - val_acc: 0.9304 - val_loss: 0.2097


2026-03-30 00:43:49.866338: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_55', 8 bytes spill stores, 8 bytes spill loads

2026-03-30 00:43:50.205100: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_43', 16 bytes spill stores, 16 bytes spill loads



Val acc (best): 0.9256  |  Val loss: 0.1958
Test acc: 0.9233  |  Test F1: 0.9234
Time: 00:08:06
SUBSET_FRAC = 1.0
  Train: 35000, Val: 5000, Test: 10000
Epoch 1/3
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 225s 44ms/step - acc: 0.8937 - loss: 0.2572 - val_acc: 0.9310 - val_loss: 0.1833
Epoch 2/3
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 190s 43ms/step - acc: 0.9370 - loss: 0.1690 - val_acc: 0.9302 - val_loss: 0.1849
Epoch 3/3
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 190s 43ms/step - acc: 0.9560 - loss: 0.1248 - val_acc: 0.9334 - val_loss: 0.1888


2026-03-30 00:54:25.028924: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_55', 8 bytes spill stores, 8 bytes spill loads

2026-03-30 00:54:25.311802: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_43', 16 bytes spill stores, 16 bytes spill loads



Val acc (best): 0.9310  |  Val loss: 0.1833
Test acc: 0.9216  |  Test F1: 0.9238
Time: 00:10:04
  Frac   ValAcc  ValLoss  TestAcc   TestF1       Time
  0.25   0.9152   0.2249   0.9120   0.9128   00:03:25
  0.50   0.9244   0.1940   0.9178   0.9170   00:05:56
  0.75   0.9256   0.1958   0.9233   0.9234   00:08:06
  1.00   0.9310   0.1833   0.9216   0.9238   00:10:04

Best: SUBSET_FRAC=1.0, ValAcc=0.9310


### Graded Questions

In [10]:
# Set a2a to the validation accuracy at min validation loss for your best configuration found in this problem
# (Yes, it is probably at 1.0!)

a2a = best_p2["val_acc"]

In [11]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a2a = {a2a:.4f}')

a2a = 0.9310


#### Question a2b:

Summarize what you observed as dataset size increased. Given that validation metrics are typically reliable to only about two decimal places, do the performance gains justify using the entire dataset? What trade-offs between accuracy and computation time did you notice?

#### Your Answer Here:

As dataset size increased from 25% to 100%, validation accuracy improved steadily: 0.9152 to 0.9244 to 0.9256 to 0.9310. The largest single jump came from 0.25 to 0.50 while the gain from 0.75 to 1.0 was only ~0.5%.

Given that validation metrics are reliable to about two decimal places, the differences between 0.75 and 1.0 are near the noise floor and may not represent a meaningful improvement. However, training time scaled roughly linearly.

Trade-off: Using 50% of the data captures most of the performance benefit at roughly half the compute cost of the full dataset. The full dataset is justified for a final model submission but for iterative experimentation and hyperparameter tuning, 25–50% is the sweet spot. The pre-trained BERT representations are already strong so fine-tuning requires less labeled data than training from scratch would.

# Problem 3 — Model swap: speed vs. accuracy (why: capacity matters)

In this problem we will compare encoder-only backbones of different sizes.

**Setup.** Keep the best `MAX_LEN`, `LR`, and `SUBSET_FRAC` from Problems 1–2. Only change the model/preset:

* **DistilBERT** (current baseline)
* **BERT-base** (larger/usually stronger)

**How to switch (two lines each).**

* DistilBERT:

  ```python
  preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset("distil_bert_base_en_uncased", sequence_length=MAX_LEN)
  model  = kh.models.DistilBertTextClassifier.from_preset("distil_bert_base_en_uncased", num_classes=2, preprocessor=preproc)
  ```

* BERT-base:

  ```python
  preproc = kh.models.BertTextClassifierPreprocessor.from_preset("bert_base_en_uncased", sequence_length=MAX_LEN)
  model  = kh.models.BertTextClassifier.from_preset("bert_base_en_uncased", num_classes=2, preprocessor=preproc)
  ```

**What to do.**

1. Train/evaluate each model once with identical settings.
2. Observe the performance metrics for each.
3. Answer the graded questions.



In [12]:
# Problem 3:
# Use best settings from Problems 1-2

BEST_MAX_LEN  = best_p1["max_len"]
BEST_LR       = best_p1["lr"]
BEST_BATCH    = best_p1["batch"]
BEST_FRAC     = best_p2["frac"]
EPOCHS        = 3
SEED          = 42

# Rebuild data with best SUBSET_FRAC
keras.utils.set_random_seed(SEED)
if 0.0 < BEST_FRAC < 1.0:
    sub = all_ds.train_test_split(train_size=BEST_FRAC, seed=SEED, stratify_by_column="label")
    ds_pool_p3 = sub["train"]
else:
    ds_pool_p3 = all_ds

splits = ds_pool_p3.train_test_split(test_size=0.20, seed=SEED, stratify_by_column="label")
train_val_pool_p3, test_ds_p3 = splits["train"], splits["test"]
splits2 = train_val_pool_p3.train_test_split(test_size=0.125, seed=SEED, stratify_by_column="label")
train_ds_p3, val_ds_p3 = splits2["train"], splits2["test"]

X_tr3 = np.array(train_ds_p3["text"], dtype=object); y_tr3 = np.array(train_ds_p3["label"], dtype="int32")
X_va3 = np.array(val_ds_p3["text"],   dtype=object); y_va3 = np.array(val_ds_p3["label"],   dtype="int32")
X_te3 = np.array(test_ds_p3["text"],  dtype=object); y_te3 = np.array(test_ds_p3["label"],  dtype="int32")

models_to_test = [
    {
        "name": "DistilBERT",
        "build": lambda: (
            kh.models.DistilBertTextClassifierPreprocessor.from_preset(
                "distil_bert_base_en_uncased", sequence_length=BEST_MAX_LEN),
            lambda pp: kh.models.DistilBertTextClassifier.from_preset(
                "distil_bert_base_en_uncased", num_classes=2, preprocessor=pp)
        ),
    },
    {
        "name": "BERT-base",
        "build": lambda: (
            kh.models.BertTextClassifierPreprocessor.from_preset(
                "bert_base_en_uncased", sequence_length=BEST_MAX_LEN),
            lambda pp: kh.models.BertTextClassifier.from_preset(
                "bert_base_en_uncased", num_classes=2, preprocessor=pp)
        ),
    },
]

p3_results = []

for model_cfg in models_to_test:
    print(f"Model: {model_cfg['name']}")

    keras.utils.set_random_seed(SEED)

    preproc, model_fn = model_cfg["build"]()
    mdl = model_fn(preproc)

    mdl.compile(
        optimizer=keras.optimizers.Adam(BEST_LR),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )

    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]
    start = time.time()
    hist = mdl.fit(
        X_tr3, y_tr3,
        validation_data=(X_va3, y_va3),
        epochs=EPOCHS,
        batch_size=BEST_BATCH,
        callbacks=cb,
        verbose=1,
    )
    elapsed = time.time() - start

    best_idx = np.argmin(hist.history["val_loss"])
    best_val_acc  = hist.history["val_acc"][best_idx]
    best_val_loss = hist.history["val_loss"][best_idx]

    logits = mdl.predict(X_te3, batch_size=EVAL_BATCH, verbose=0)
    y_pred = logits.argmax(axis=-1)
    acc_metric = evaluate.load("accuracy")
    f1_metric  = evaluate.load("f1")
    test_acc = acc_metric.compute(predictions=y_pred, references=y_te3)["accuracy"]
    test_f1  = f1_metric.compute(predictions=y_pred, references=y_te3)["f1"]

    p3_results.append({
        "name": model_cfg["name"],
        "val_acc": best_val_acc,
        "val_loss": best_val_loss,
        "test_acc": test_acc,
        "test_f1": test_f1,
        "time": elapsed,
    })

    print(f"Val acc (best): {best_val_acc:.4f}  |  Val loss: {best_val_loss:.4f}")
    print(f"Test acc: {test_acc:.4f}  |  Test F1: {test_f1:.4f}")
    print(f"Time: {time.strftime('%H:%M:%S', time.gmtime(elapsed))}")

# Summary
print(f"{'Model':>15} {'ValAcc':>8} {'ValLoss':>8} {'TestAcc':>8} {'TestF1':>8} {'Time':>10}")
for r in p3_results:
    print(f"{r['name']:>15} {r['val_acc']:>8.4f} {r['val_loss']:>8.4f} {r['test_acc']:>8.4f} {r['test_f1']:>8.4f} {time.strftime('%H:%M:%S', time.gmtime(r['time'])):>10}")

best_p3 = max(p3_results, key=lambda r: r["val_acc"])
print(f"\nBest model: {best_p3['name']}, ValAcc={best_p3['val_acc']:.4f}")

Model: DistilBERT
Epoch 1/3
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 221s 44ms/step - acc: 0.8938 - loss: 0.2572 - val_acc: 0.9314 - val_loss: 0.1833
Epoch 2/3
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 195s 45ms/step - acc: 0.9371 - loss: 0.1691 - val_acc: 0.9298 - val_loss: 0.1849
Epoch 3/3
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 188s 43ms/step - acc: 0.9561 - loss: 0.1249 - val_acc: 0.9338 - val_loss: 0.1895
Val acc (best): 0.9314  |  Val loss: 0.1833
Test acc: 0.9215  |  Test F1: 0.9237
Time: 00:10:04
Model: BERT-base
Resuming download from 283115520 bytes (155220528 bytes left)...
Resuming download to /home/luan/.cache/kagglehub/models/keras/bert/keras/bert_base_en_uncased/3/model.weights.h5 (283115520/438336048) bytes left.


100%|██████████| 418M/418M [00:02<00:00, 77.5MB/s]


Epoch 1/3
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 416s 83ms/step - acc: 0.9058 - loss: 0.2349 - val_acc: 0.9374 - val_loss: 0.1654
Epoch 2/3
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 361s 83ms/step - acc: 0.9523 - loss: 0.1360 - val_acc: 0.9368 - val_loss: 0.1819
Epoch 3/3
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 360s 82ms/step - acc: 0.9730 - loss: 0.0830 - val_acc: 0.9460 - val_loss: 0.1686
Val acc (best): 0.9374  |  Val loss: 0.1654
Test acc: 0.9297  |  Test F1: 0.9316
Time: 00:18:58
          Model   ValAcc  ValLoss  TestAcc   TestF1       Time
     DistilBERT   0.9314   0.1833   0.9215   0.9237   00:10:04
      BERT-base   0.9374   0.1654   0.9297   0.9316   00:18:58

Best model: BERT-base, ValAcc=0.9374


### Graded Questions

In [13]:
# Set a1a to the validation accuracy at min validation loss for your best model found in this problem

a3a = best_p3["val_acc"]

In [14]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a3a = {a3a:.4f}')

a3a = 0.9374


#### Question a3b:

**Answer briefly.**

* Which model gives the best **accuracy/F1**?
* Which is **fastest** per epoch?
* Given limited development time or compute resources, which model is the best **overall choice** and why?

#### Your Answer Here:

Best accuracy/F1: BERT-base achieved the highest validation accuracy and test F1. It outperformed DistilBERT by about 0.6–0.8 percentage points. This is expected since BERT-base has ~110M parameters with 12 transformer layers vs. DistilBERT's ~66M parameters and 6 layers.

Fastest per epoch: DistilBERT was nearly 2× faster, finishing in ~10 minutes vs. ~19 minutes for BERT-base. With half the transformer layers, its forward and backward passes are substantially cheaper.

Best overall choice: DistilBERT is the best overall choice. It achieves accuracy within ~0.6% of BERT-base while training nearly twice as fast. This makes it ideal for rapid experimentation and deployment in resource-constrained environments. The small accuracy gap rarely justifies the nearly doubled compute cost, especially during iterative development. BERT-base would only be preferred when every fraction of a percent matters for a final production model.